# Importações

In [1]:
import pandas as pd
import os
import numpy as np
from sqlalchemy import text
from sqlalchemy import create_engine

PRPI_DATABASE_URL = os.getenv("PRPI_DATABASE_URL")
engine = create_engine(PRPI_DATABASE_URL)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

# Carregamento de dados e pré transformações

## Produções Técnicas

In [2]:
query = "select * from cnpq_producaotecnica order by tipo_pub asc;"
producoes_tecnicas_df = pd.read_sql(query, engine)

producoes_tecnicas_df = producoes_tecnicas_df.rename(columns={
    "id": "ID",
    "curriculo_id": "ID do Currículo",
    "sequencia": "Sequência",
    "titulo": "Título",
    "ano": "Ano",
    "pais": "País",
    "idioma": "Idioma",
    "flag_relevancia": "Relevância",
    "doi": "DOI",
    "tipo_pub": "Tipo de Publicação",
    "informacao_adicional_id": "ID de Informação Adicional"
})

### Padronização de Tipo de Publicação

In [3]:
producoes_tecnicas_df["Tipo de Publicação"] = (
    producoes_tecnicas_df["Tipo de Publicação"]
        .str.replace("-", " ", regex=False)
        .str.lower()
        .str.replace("cao", "ção", regex=False)
        .str.title()
)

## Membros de Projetos

In [4]:
query = """
  SELECT
  pr.id AS projeto_id,
  pr.titulo AS titulo_projeto,
  p.nome AS nome_pessoa,
  p.email AS email_pessoa,
  uo.nome AS campus,
  CASE
      WHEN p.email ILIKE '%@aluno.ifce.edu.br' THEN 'Discente'
      WHEN s.eh_docente = TRUE THEN 'Docente'
      WHEN s.eh_tecnico_administrativo = TRUE THEN 'TAE'
      ELSE NULL
  END AS categoria_funcional,
  pp.vinculo,
  pp.ativo,
  pr.inicio_execucao,
  EXTRACT(YEAR FROM pr.inicio_execucao) AS ano_inicio_execucao,
  pr.area_conhecimento_id,
  ac.descricao AS descricao_area_conhecimento,
  pr.edital_id,
  pe.titulo AS titulo_edital,
  pe.tipo_edital,
  -- Descrição do tipo do edital
  CASE pe.tipo_edital
      WHEN '1' THEN 'Pesquisa'
      WHEN '2' THEN 'Inovação'
      WHEN '3' THEN 'Pesquisa/Inovação - Contínuo'
      ELSE 'Desconhecido'
  END AS descricao_tipo_edital
FROM pesquisa_projeto pr
JOIN pesquisa_participacao pp
  ON pp.projeto_id = pr.id
JOIN unidadeorganizacional uo
  ON uo.id = pr.uo_id
JOIN comum_vinculo cv
  ON cv.id = pp.vinculo_pessoa_id
JOIN pessoa p
  ON p.id = cv.pessoa_id
LEFT JOIN servidor s
  ON s.pessoa_fisica_id = p.id
LEFT JOIN rh_areaconhecimento ac
  ON ac.id = pr.area_conhecimento_id
LEFT JOIN pesquisa_edital pe
  ON pe.id = pr.edital_id
ORDER BY pr.id, p.nome;
"""

with engine.connect() as conn:
    membros_projetos_df = pd.read_sql(text(query), conn)
    
membros_projetos_df = membros_projetos_df.rename(columns={
    "projeto_id": "ID do Projeto",
    "titulo_projeto": "Título do Projeto",
    "nome_pessoa": "Nome do Membro",
    "email_pessoa": "Email do Membro",
    "campus": "Campus",
    "categoria_funcional": "Categoria do Membro",
    "vinculo": "Vínculo com Projeto",
    "ativo": "Ativo",
    "inicio_execucao": "Início da Execução",
    "ano_inicio_execucao": "Ano de Início da Execução",
    "area_conhecimento_id": "ID da Área de Conhecimento",
    "descricao_area_conhecimento": "Área de Conhecimento",
    "edital_id": "ID do Edital",
    "titulo_edital": "Título do Edital",
    "tipo_edital": "ID do Tipo de Edital",
    "descricao_tipo_edital": "Tipo de Edital"
})

### Padronização de Edital

In [5]:
membros_projetos_df["Título do Edital"] = membros_projetos_df["Título do Edital"].str.upper()

conditions = [
    membros_projetos_df["Título do Edital"].str.startswith("PROGRAMA DE INICIAÇÃO CIENTÍFICA E TECNOLÓGICA VOLUNTÁRIA", na=False),
    membros_projetos_df["Título do Edital"].str.startswith("PIBITI", na=False),
    membros_projetos_df["Título do Edital"].str.startswith("PIBIC JR AF", na=False),
    membros_projetos_df["Título do Edital"].str.startswith("PIBIC JR", na=False),
    membros_projetos_df["Título do Edital"].str.startswith("PIBIC AF", na=False),
    membros_projetos_df["Título do Edital"].str.startswith("PIBIC", na=False),
]

choices = [
    "PICTV",
    "PIBITI",
    "PIBIC JR AF",
    "PIBIC JR",
    "PIBIC AF",
    "PIBIC",
]

membros_projetos_df["Edital"] = np.select(conditions, choices, default="FOMENTO EXTERNO")
membros_projetos_df = membros_projetos_df[membros_projetos_df["Título do Edital"] != "."]

### Padronização da Categoria do Membro

In [6]:
membros_projetos_df[["Categoria do Membro"]] = (
    membros_projetos_df[["Categoria do Membro"]]
        .fillna("Não Informado")
)

### Vinculo de Pesquisadores

In [7]:
vinculo_pesquisadores_df = membros_projetos_df

vinculo_pesquisadores_df["Tem Bolsista"] = vinculo_pesquisadores_df["Vínculo com Projeto"].eq("Bolsista")
vinculo_pesquisadores_df["Tem Voluntario"] = vinculo_pesquisadores_df["Vínculo com Projeto"].eq("Voluntário")

vinculo_pesquisadores_df = (
    vinculo_pesquisadores_df.groupby(["Nome do Membro", "Categoria do Membro"]).agg({
          "Tem Bolsista": "max",
          "Tem Voluntario": "max"
      })
      .reset_index()
)

vinculo_pesquisadores_df["Tipo de Participação"] = (
    vinculo_pesquisadores_df["Tem Bolsista"].map({True: "Bolsista", False: ""}) +
    vinculo_pesquisadores_df["Tem Voluntario"].map({True: "Voluntário", False: ""})
).str.replace("BolsistaVoluntário", "Bolsista e Voluntário")

display(vinculo_pesquisadores_df)

,Nome do Membro,Categoria do Membro,Tem Bolsista,Tem Voluntario,Tipo de Participação
0,ANA LETÍCIA ARAUJO NUNES,Discente,True,False,Bolsista
1,ANTÔNIO HEITOR MARTINS VALENTIM,Discente,True,False,Bolsista
2,ARTHUR HERBSTER FERNANDES VOGEL,Discente,True,False,Bolsista
3,Aaron Keven Ferreira Rocha,Discente,True,False,Bolsista
4,Aaron Vinicius Souza Angelim,Discente,False,True,Voluntário
5,Abdias Araujo Bezerra,Discente,True,False,Bolsista
6,Abigail Luz Nobre,Não Informado,False,True,Voluntário
7,Abimaell Jonathan Siqueira de Souza,Discente,False,True,Voluntário
8,Abner Jose Girao Meneses,Docente,True,True,Bolsista e Voluntário
9,Abraão Alves da Silva,Discente,True,False,Bolsista


## Projetos de Pesquisa

In [8]:
query = """
SELECT DISTINCT
 projeto.id AS cod_projeto,
 projeto.titulo AS titulo_projeto,
 uo.nome AS campus,
 projeto.aprovado, 
 projeto.inicio_execucao,
 EXTRACT(YEAR FROM projeto.inicio_execucao) AS ano_inicio_execucao,
 projeto.fim_execucao,
 EXTRACT(YEAR FROM projeto.fim_execucao) AS ano_fim_execucao,
 projeto.data_finalizacao_conclusao,
 gp.descricao as grupo_pesquisa_nome,
 ac.descricao as area_conhecimento,
 COALESCE(ac_superior.descricao, ac.descricao) AS grande_area_conhecimento,
 projeto.edital_id,
 edital.titulo AS titulo_edital,
 edital.tipo_edital as tipo_edital_id,
 CASE
     WHEN edital.tipo_edital = '1' THEN 'PESQUISA'
     WHEN edital.tipo_edital = '2' THEN 'INOVACAO'
     WHEN edital.tipo_edital = '3' THEN 'PESQUISA_INOVACAO_CONTINUO'
     WHEN edital.tipo_edital = '4' THEN 'EXTENSAO_FLUXO_CONTINUO'
     ELSE 'DESCONHECIDO'
 END AS descricao_tipo_edital 
FROM
 pesquisa_projeto projeto
JOIN
 pesquisa_edital edital ON projeto.edital_id = edital.id
LEFT JOIN
 cnpq_grupopesquisa gp ON projeto.grupo_pesquisa_id = gp.id
LEFT JOIN
 rh_areaconhecimento ac ON projeto.area_conhecimento_id = ac.id
LEFT JOIN
 rh_areaconhecimento ac_superior ON ac.superior_id = ac_superior.id
LEFT JOIN 
 unidadeorganizacional uo ON uo.id = projeto.uo_id
ORDER BY
 cod_projeto ASC;
"""

with engine.connect() as conn:
    projetos_df = pd.read_sql(text(query), conn)

projetos_df = projetos_df.rename(columns={
     "cod_projeto": "ID do Projeto",
    "titulo_projeto": "Título do Projeto",
    "campus": "Campus",
    "aprovado": "Aprovado",
    "inicio_execucao": "Início da Execução",
    "ano_inicio_execucao": "Ano de Início da Execução",
    "fim_execucao": "Fim da Execução",
    "ano_fim_execucao": "Ano de Fim da Execução",
    "grupo_pesquisa_nome": "Nome do Grupo de Pesquisa",
    "area_conhecimento": "Área de Conhecimento",
    "edital_id": "ID do Edital",
    "titulo_edital": "Título do Edital",
    "tipo_edital_id": "ID do Tipo de Edital",
    "descricao_tipo_edital": "Tipo de Edital",
    "grande_area_conhecimento": "Grande Área de Conhecimento",
    "data_finalizacao_conclusao": "Data de Finalização da Conclusão"
})

### Padronização de Campus do Projeto

In [9]:
projetos_df["Campus"] = projetos_df["Campus"].str.replace("CAMPUS ", "", regex=False)
projetos_df["Campus"] = projetos_df["Campus"].str.replace("AVANÇADO DE ", "", regex=False)

display(projetos_df["Campus"].unique())

<StringArray>
[         'REITORIA',         'FORTALEZA',           'QUIXADA',
           'CRATEUS',           'TIANGUA',         'ITAPIPOCA',
 'LIMOEIRO DO NORTE',            'IGUATU',           'CAMOCIM',
 'JUAZEIRO DO NORTE',            'ACARAU',         'MARACANAÚ',
            'SOBRAL',          'BATURITE',        'MARANGUAPE',
           'CAUCAIA',             'CEDRO',         'HORIZONTE',
         ' ACOPIARA',           'ARACATI',       'MORADA NOVA',
           'UBAJARA',         'JAGUARIBE',             'PECÉM',
              'TAUA',           'CANINDE',             'CRATO',
         'TABULEIRO',          'PARACURU',        'BOA VIAGEM',
            'UMIRIM',      'GUARAMIRANGA',        'JAGUARUANA',
           'MOMBAÇA']
Length: 34, dtype: str

### Situação do Projeto

In [10]:
conditions = [
    projetos_df["Aprovado"] == False,
    projetos_df["Data de Finalização da Conclusão"].isna()
]

choices = ["Não Aprovado", "Em Execução"]

projetos_df["Situação do Projeto"] = np.select(
    conditions,
    choices,
    default="Concluído"
)

### Padronização de Título do Projeto

In [11]:
projetos_df['Título do Projeto'] = (
    projetos_df['Título do Projeto']
    .astype(str)
    .str.strip()
    .str.title()
    .str.replace(r'\s+', ' ', regex=True)
)

### Padronização de Edital

In [12]:
projetos_df["Título do Edital"] = projetos_df["Título do Edital"].str.upper()

conditions = [
    projetos_df["Título do Edital"].str.startswith("PROGRAMA DE INICIAÇÃO CIENTÍFICA E TECNOLÓGICA VOLUNTÁRIA", na=False),
    projetos_df["Título do Edital"].str.startswith("PIBITI", na=False),
    projetos_df["Título do Edital"].str.startswith("PIBIC JR AF", na=False),
    projetos_df["Título do Edital"].str.startswith("PIBIC JR", na=False),
    projetos_df["Título do Edital"].str.startswith("PIBIC AF", na=False),
    projetos_df["Título do Edital"].str.startswith("PIBIC", na=False),
]

choices = [
    "PICTV",
    "PIBITI",
    "PIBIC JR AF",
    "PIBIC JR",
    "PIBIC AF",
    "PIBIC",
]

projetos_df["Edital"] = np.select(conditions, choices, default="FOMENTO EXTERNO")
projetos_df = projetos_df[projetos_df["Título do Edital"] != "."]

## Projetos Aprovados por Grupo de Pesquisa

In [13]:
query = """
SELECT 
    gp.id as grupo_pesquisa_id,
    gp.descricao as grupo_pesquisa_nome,
    COUNT(*) AS total
FROM pesquisa_projeto as pp
JOIN cnpq_grupopesquisa as gp
	ON pp.grupo_pesquisa_id = gp.id
WHERE aprovado = TRUE
GROUP BY gp.id, gp.descricao
ORDER BY gp.id;
"""

with engine.connect() as conn:
    grupos_pesquisa_df = pd.read_sql(text(query), conn)

grupos_pesquisa_df = grupos_pesquisa_df.rename(columns={
    "grupo_pesquisa_id": "ID do Grupo de Pesquisa",
    "grupo_pesquisa_nome": "Nome do Grupo de Pesquisa",
    "total": "Total de Projetos Aprovados"
})

# Geral ✅

## Estatísticas

In [17]:
total_projetos = projetos_df["Título do Projeto"].nunique()
total_projetos_aprovados = projetos_df[projetos_df["Aprovado"] == True].shape[0]
total_areas_conhecimento = projetos_df["Área de Conhecimento"].nunique()
total_grupos_de_pesquisa = projetos_df["Nome do Grupo de Pesquisa"].nunique()
total_patentes = producoes_tecnicas_df[producoes_tecnicas_df["Tipo de Publicação"] == "Patente"].shape[0]
total_producoes_tecnicas = producoes_tecnicas_df.shape[0]

print(f"Total de Projetos: {total_projetos}")
print(f"Total de Projetos Aprovados: {total_projetos_aprovados}")
print(f"Total de Áreas de Conhecimento: {total_areas_conhecimento}")
print(f"Total de Grupos de Pesquisa: {total_grupos_de_pesquisa}")
print(f"Total de Patentes: {total_patentes}")
print(f"Total de Produções Técnicas: {total_producoes_tecnicas}")

Total de Projetos: 2856
Total de Projetos Aprovados: 1811
Total de Áreas de Conhecimento: 66
Total de Grupos de Pesquisa: 195
Total de Patentes: 544
Total de Produções Técnicas: 82496


## Pesquisadores

In [18]:
total_pesquisadores = vinculo_pesquisadores_df["Nome do Membro"].count()
print(f"Total de Pesquisadores: {total_pesquisadores}")

analise_pesquisadores = vinculo_pesquisadores_df.groupby("Categoria do Membro").agg(
    total_pesquisadores=('Nome do Membro', 'count'),
    bolsistas=('Tipo de Participação', lambda x: (x == 'Bolsista').sum()),
    voluntarios=('Tipo de Participação', lambda x: (x == 'Voluntário').sum()),
    bolsistas_voluntarios=('Tipo de Participação', lambda x: (x == 'Bolsista e Voluntário').sum())
).reset_index().rename(columns={'total_pesquisadores': 'Total de Pesquisadores', 'bolsistas': 'Bolsistas', 'voluntarios': 'Voluntários', 'bolsistas_voluntarios': 'Bolsistas e Voluntários'})

analise_pesquisadores = analise_pesquisadores.sort_values(by="Total de Pesquisadores", ascending=False)

analise_pesquisadores['Bolsistas'] = (
    analise_pesquisadores['Bolsistas'].astype(str) +
    ' (' + (analise_pesquisadores['Bolsistas'] / analise_pesquisadores['Total de Pesquisadores'] * 100).round(2).astype(str) + '%)'
)

analise_pesquisadores['Voluntários'] = (
    analise_pesquisadores['Voluntários'].astype(str) +
    ' (' + (analise_pesquisadores['Voluntários'] / analise_pesquisadores['Total de Pesquisadores'] * 100).round(2).astype(str) + '%)'
)

analise_pesquisadores['Bolsistas e Voluntários'] = (
    analise_pesquisadores['Bolsistas e Voluntários'].astype(str) +
    ' (' + (analise_pesquisadores['Bolsistas e Voluntários'] / analise_pesquisadores['Total de Pesquisadores'] * 100).round(2).astype(str) + '%)'
)

display(analise_pesquisadores)

Total de Pesquisadores: 3390


,Categoria do Membro,Total de Pesquisadores,Bolsistas,Voluntários,Bolsistas e Voluntários
0,Discente,2211,1136 (51.38%),910 (41.16%),165 (7.46%)
1,Docente,1006,51 (5.07%),777 (77.24%),178 (17.69%)
3,TAE,94,34 (36.17%),52 (55.32%),8 (8.51%)
2,Não Informado,79,28 (35.44%),48 (60.76%),3 (3.8%)


# Projetos Por Grande Área ✅

## Projetos por Grande Área

In [ ]:
analise_grande_area = projetos_df.groupby("Grande Área de Conhecimento").agg(
    total_projetos=("ID do Projeto", "count")
).reset_index().rename(columns={"total_projetos": "Total de Projetos"}).sort_values(by="Total de Projetos", ascending=False)

analise_grande_area['Total de Projetos'] = (
    analise_grande_area['Total de Projetos'].astype(str) + 
    ' (' + (analise_grande_area['Total de Projetos'] / analise_grande_area['Total de Projetos'].sum() * 100).round(2).astype(str) + '%)'
)

display(analise_grande_area)

   Grande Área de Conhecimento Total de Projetos
3   CIÊNCIAS EXATAS E DA TERRA       840 (24.9%)
6                  ENGENHARIAS      686 (20.33%)
0            CIÊNCIAS AGRÁRIAS      624 (18.49%)
8             MULTIDISCIPLINAR       302 (8.95%)
4             CIÊNCIAS HUMANAS       294 (8.71%)
7  LINGUÍSTICA, LETRAS E ARTES       178 (5.28%)
2            CIÊNCIAS DA SAÚDE       168 (4.98%)
1          CIÊNCIAS BIOLÓGICAS       143 (4.24%)
5   CIÊNCIAS SOCIAIS APLICADAS       139 (4.12%)


## Grande Área por Ano

In [ ]:
analise_grande_area_por_ano = projetos_df.groupby("Grande Área de Conhecimento").agg(
    total_2022=("Ano de Início da Execução", lambda x: (x == 2022).sum()),
    total_2023=("Ano de Início da Execução", lambda x: (x == 2023).sum()),
    total_2024=("Ano de Início da Execução", lambda x: (x == 2024).sum()),
    total_2025=("Ano de Início da Execução", lambda x: (x == 2025).sum()),
    total_2026=("Ano de Início da Execução", lambda x: (x == 2026).sum())
).reset_index().rename(columns={"total_2026": "2026", "total_2025": "2025", "total_2024": "2024", "total_2023": "2023", "total_2022": "2022"})

display(analise_grande_area_por_ano)

,Grande Área de Conhecimento,2022,2023,2024,2025,2026
0,CIÊNCIAS AGRÁRIAS,0,180,202,231,10
1,CIÊNCIAS BIOLÓGICAS,0,37,37,62,7
2,CIÊNCIAS DA SAÚDE,0,47,56,63,2
3,CIÊNCIAS EXATAS E DA TERRA,1,244,265,312,18
4,CIÊNCIAS HUMANAS,0,75,75,138,6
5,CIÊNCIAS SOCIAIS APLICADAS,2,38,32,65,2
6,ENGENHARIAS,1,215,218,245,7
7,"LINGUÍSTICA, LETRAS E ARTES",0,55,47,72,4
8,MULTIDISCIPLINAR,1,80,79,138,4


## Situação de Projetos por Campus

In [ ]:
analise_projetos_por_campus = projetos_df.groupby("Campus").agg(
    total_projetos=("ID do Projeto", "count"),
    concluidos=("Situação do Projeto", lambda x: (x == "Concluído").sum()),
    em_execucao=("Situação do Projeto", lambda x: (x == "Em Execução").sum()),
    nao_aprovados=("Situação do Projeto", lambda x: (x == "Não Aprovado").sum())
).reset_index().rename(columns={"total_projetos": "Total de Projetos", "concluidos": "Concluídos", "em_execucao": "Em Execução", "nao_aprovados": "Não Aprovados"})

analise_projetos_por_campus = analise_projetos_por_campus.sort_values(by="Total de Projetos", ascending=False)

analise_projetos_por_campus['Concluídos'] = (
    analise_projetos_por_campus['Concluídos'].astype(str) +
    ' (' + (analise_projetos_por_campus['Concluídos'] / analise_projetos_por_campus['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_projetos_por_campus['Em Execução'] = (
    analise_projetos_por_campus['Em Execução'].astype(str) +
    ' (' + (analise_projetos_por_campus['Em Execução'] / analise_projetos_por_campus['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_projetos_por_campus['Não Aprovados'] = (
    analise_projetos_por_campus['Não Aprovados'].astype(str) +
    ' (' + (analise_projetos_por_campus['Não Aprovados'] / analise_projetos_por_campus['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

display(analise_projetos_por_campus)

## Grande Área por Campus

In [ ]:
analise_grande_area_por_campus = projetos_df.groupby("Campus").agg(
    total_projetos=("ID do Projeto", "count"),
    total_ciências_exatas_e_terra=("Grande Área de Conhecimento", lambda x: (x == "CIÊNCIAS EXATAS E DA TERRA").sum()),
    total_ciências_biologicas=("Grande Área de Conhecimento", lambda x: (x == "CIÊNCIAS BIOLÓGICAS").sum()),    
    total_ciências_humanas=("Grande Área de Conhecimento", lambda x: (x == "CIÊNCIAS HUMANAS").sum()),
    total_ciências_sociais_aplicadas=("Grande Área de Conhecimento", lambda x: (x == "CIÊNCIAS SOCIAIS APLICADAS").sum()),
    total_engenharia=("Grande Área de Conhecimento", lambda x: (x == "ENGENHARIAS").sum()),
    total_agrarias=("Grande Área de Conhecimento", lambda x: (x == "CIÊNCIAS AGRÁRIAS").sum()),
    total_saude=("Grande Área de Conhecimento", lambda x: (x == "CIÊNCIAS DA SAÚDE").sum()),
    total_multidisciplinar=("Grande Área de Conhecimento", lambda x: (x == "MULTIDISCIPLINAR").sum()),
    total_linguistica=("Grande Área de Conhecimento", lambda x: (x == "LINGUÍSTICA, LETRAS E ARTES").sum())
).reset_index().rename(columns={
    "total_projetos": "Total de Projetos", 
    "total_ciências_exatas_e_terra": "Ciências Exatas e da Terra", 
    "total_ciências_biologicas": "Ciências Biológicas", 
    "total_ciências_humanas": "Ciências Humanas", 
    "total_ciências_sociais_aplicadas": "Ciências Sociais Aplicadas", 
    "total_engenharia": "Engenharia", 
    "total_agrarias": "Ciências Agrárias", 
    "total_saude": "Ciências da Saúde", 
    "total_multidisciplinar": "Multidisciplinar",
    "total_linguistica": "Linguística, Letras e Artes"
})

analise_grande_area_por_campus = analise_grande_area_por_campus.sort_values(by="Total de Projetos", ascending=False)

analise_grande_area_por_campus['Ciências Exatas e da Terra'] = (
    analise_grande_area_por_campus['Ciências Exatas e da Terra'].astype(str) +
    ' (' + (analise_grande_area_por_campus['Ciências Exatas e da Terra'] / analise_grande_area_por_campus['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_por_campus['Ciências Biológicas'] = (
    analise_grande_area_por_campus['Ciências Biológicas'].astype(str) +
    ' (' + (analise_grande_area_por_campus['Ciências Biológicas'] / analise_grande_area_por_campus['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_por_campus['Ciências Humanas'] = (
    analise_grande_area_por_campus['Ciências Humanas'].astype(str) +
    ' (' + (analise_grande_area_por_campus['Ciências Humanas'] / analise_grande_area_por_campus['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_por_campus['Ciências Sociais Aplicadas'] = (
    analise_grande_area_por_campus['Ciências Sociais Aplicadas'].astype(str) +
    ' (' + (analise_grande_area_por_campus['Ciências Sociais Aplicadas'] / analise_grande_area_por_campus['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)   

analise_grande_area_por_campus['Engenharia'] = (
    analise_grande_area_por_campus['Engenharia'].astype(str) +
    ' (' + (analise_grande_area_por_campus['Engenharia'] / analise_grande_area_por_campus['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_por_campus['Ciências Agrárias'] = (
    analise_grande_area_por_campus['Ciências Agrárias'].astype(str) +
    ' (' + (analise_grande_area_por_campus['Ciências Agrárias'] / analise_grande_area_por_campus['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_por_campus['Ciências da Saúde'] = (
    analise_grande_area_por_campus['Ciências da Saúde'].astype(str) +
    ' (' + (analise_grande_area_por_campus['Ciências da Saúde'] / analise_grande_area_por_campus['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_por_campus['Multidisciplinar'] = (
    analise_grande_area_por_campus['Multidisciplinar'].astype(str) +
    ' (' + (analise_grande_area_por_campus['Multidisciplinar'] / analise_grande_area_por_campus['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_por_campus['Linguística, Letras e Artes'] = (
    analise_grande_area_por_campus['Linguística, Letras e Artes'].astype(str) +
    ' (' + (analise_grande_area_por_campus['Linguística, Letras e Artes'] / analise_grande_area_por_campus['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

display(analise_grande_area_por_campus)

,Campus,Total de Projetos,Ciências Exatas e da Terra,Ciências Biológicas,Ciências Humanas,Ciências Sociais Aplicadas,Engenharia,Ciências Agrárias,Ciências da Saúde,Multidisciplinar,"Linguística, Letras e Artes"
11,FORTALEZA,362,74 (20.44%),5 (1.38%),30 (8.29%),13 (3.59%),167 (46.13%),10 (2.76%),18 (4.97%),25 (6.91%),20 (5.52%)
19,LIMOEIRO DO NORTE,329,29 (8.81%),2 (0.61%),32 (9.73%),5 (1.52%),51 (15.5%),125 (37.99%),46 (13.98%),12 (3.65%),27 (8.21%)
28,SOBRAL,210,9 (4.29%),7 (3.33%),4 (1.9%),5 (2.38%),28 (13.33%),139 (66.19%),1 (0.48%),15 (7.14%),2 (0.95%)
20,MARACANAÚ,194,83 (42.78%),0 (0.0%),1 (0.52%),0 (0.0%),48 (24.74%),4 (2.06%),0 (0.0%),57 (29.38%),1 (0.52%)
26,QUIXADA,191,52 (27.23%),0 (0.0%),32 (16.75%),24 (12.57%),69 (36.13%),0 (0.0%),0 (0.0%),14 (7.33%),0 (0.0%)
14,IGUATU,167,59 (35.33%),1 (0.6%),32 (19.16%),17 (10.18%),1 (0.6%),36 (21.56%),6 (3.59%),14 (8.38%),1 (0.6%)
31,TIANGUA,138,74 (53.62%),0 (0.0%),6 (4.35%),2 (1.45%),0 (0.0%),41 (29.71%),3 (2.17%),2 (1.45%),10 (7.25%)
18,JUAZEIRO DO NORTE,136,10 (7.35%),10 (7.35%),2 (1.47%),2 (1.47%),55 (40.44%),10 (7.35%),25 (18.38%),22 (16.18%),0 (0.0%)
6,CANINDE,133,36 (27.07%),0 (0.0%),30 (22.56%),4 (3.01%),13 (9.77%),0 (0.0%),37 (27.82%),8 (6.02%),5 (3.76%)
10,CRATO,118,7 (5.93%),1 (0.85%),8 (6.78%),0 (0.0%),3 (2.54%),78 (66.1%),2 (1.69%),5 (4.24%),14 (11.86%)


## Grande Área por Edital

In [21]:
analise_grande_area_por_edital = projetos_df.groupby("Grande Área de Conhecimento").agg(
    total_projetos=("ID do Projeto", "count"),
    total_externo=("Edital", lambda x: (x == "FOMENTO EXTERNO").sum()),
    total_pibic=("Edital", lambda x: (x == "PIBIC").sum()),
    total_pibic_af=("Edital", lambda x: (x == "PIBIC AF").sum()),
    total_pibic_jr=("Edital", lambda x: (x == "PIBIC JR").sum()),
    total_pibic_jr_af=("Edital", lambda x: (x == "PIBIC JR AF").sum()),
    total_pictv=("Edital", lambda x: (x == "PICTV").sum()),
    total_pibiti=("Edital", lambda x: (x == "PIBITI").sum())
).reset_index().rename(columns={
    "total_projetos": "Total de Projetos",
    "total_externo": "Fomento Externo",
    "total_pibic": "PIBIC",
    "total_pibic_af": "PIBIC AF",
    "total_pibic_jr": "PIBIC JR",
    "total_pibic_jr_af": "PIBIC JR AF",
    "total_pictv": "PICTV",
    "total_pibiti": "PIBITI"
})
analise_grande_area_por_edital = analise_grande_area_por_edital.sort_values(by="Total de Projetos", ascending=False)

analise_grande_area_por_edital['Fomento Externo'] = (
    analise_grande_area_por_edital['Fomento Externo'].astype(str) +
    ' (' + (analise_grande_area_por_edital['Fomento Externo'] / analise_grande_area_por_edital['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_por_edital['PIBIC'] = (
    analise_grande_area_por_edital['PIBIC'].astype(str) + 
    ' (' + (analise_grande_area_por_edital['PIBIC'] / analise_grande_area_por_edital['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_por_edital['PIBIC AF'] = (
    analise_grande_area_por_edital['PIBIC AF'].astype(str) +
    ' (' + (analise_grande_area_por_edital['PIBIC AF'] / analise_grande_area_por_edital['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_por_edital['PIBIC JR'] = (
    analise_grande_area_por_edital['PIBIC JR'].astype(str) +
    ' (' + (analise_grande_area_por_edital['PIBIC JR'] / analise_grande_area_por_edital['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_por_edital['PIBIC JR AF'] = (
    analise_grande_area_por_edital['PIBIC JR AF'].astype(str) +
    ' (' + (analise_grande_area_por_edital['PIBIC JR AF'] / analise_grande_area_por_edital['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_por_edital['PICTV'] = (
    analise_grande_area_por_edital['PICTV'].astype(str) +
    ' (' + (analise_grande_area_por_edital['PICTV'] / analise_grande_area_por_edital['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_por_edital['PIBITI'] = (
    analise_grande_area_por_edital['PIBITI'].astype(str) +
    ' (' + (analise_grande_area_por_edital['PIBITI'] / analise_grande_area_por_edital['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

display(analise_grande_area_por_edital)

,Grande Área de Conhecimento,Total de Projetos,Fomento Externo,PIBIC,PIBIC AF,PIBIC JR,PIBIC JR AF,PICTV,PIBITI
3,CIÊNCIAS EXATAS E DA TERRA,840,82 (9.76%),226 (26.9%),49 (5.83%),117 (13.93%),30 (3.57%),196 (23.33%),140 (16.67%)
6,ENGENHARIAS,685,40 (5.84%),206 (30.07%),45 (6.57%),122 (17.81%),22 (3.21%),113 (16.5%),137 (20.0%)
0,CIÊNCIAS AGRÁRIAS,624,13 (2.08%),215 (34.46%),72 (11.54%),88 (14.1%),27 (4.33%),73 (11.7%),136 (21.79%)
8,MULTIDISCIPLINAR,302,11 (3.64%),97 (32.12%),41 (13.58%),46 (15.23%),14 (4.64%),50 (16.56%),43 (14.24%)
4,CIÊNCIAS HUMANAS,294,7 (2.38%),96 (32.65%),34 (11.56%),34 (11.56%),20 (6.8%),82 (27.89%),21 (7.14%)
7,"LINGUÍSTICA, LETRAS E ARTES",178,6 (3.37%),77 (43.26%),12 (6.74%),16 (8.99%),4 (2.25%),57 (32.02%),6 (3.37%)
2,CIÊNCIAS DA SAÚDE,168,3 (1.79%),59 (35.12%),25 (14.88%),23 (13.69%),4 (2.38%),34 (20.24%),20 (11.9%)
1,CIÊNCIAS BIOLÓGICAS,143,5 (3.5%),49 (34.27%),18 (12.59%),13 (9.09%),3 (2.1%),42 (29.37%),13 (9.09%)
5,CIÊNCIAS SOCIAIS APLICADAS,139,9 (6.47%),45 (32.37%),14 (10.07%),22 (15.83%),9 (6.47%),30 (21.58%),10 (7.19%)


# Projetos por Área de Conhecimento ✅

## Total de Projetos por Área de Conhecimento

In [28]:
analise_area_conhecimento = projetos_df.groupby("Área de Conhecimento").agg(
    total_projetos=("Título do Projeto", "nunique"),
).reset_index().rename(columns={"total_projetos": "Total de Projetos"}).sort_values(by="Total de Projetos", ascending=False)

display(analise_area_conhecimento)

,Área de Conhecimento,Total de Projetos
11,CIÊNCIA DA COMPUTAÇÃO,421
30,ENGENHARIA ELÉTRICA,231
1,AGRONOMIA,205
57,QUÍMICA,173
34,ENGENHARIA SANITÁRIA,164
13,CIÊNCIA E TECNOLOGIA DE ALIMENTOS,160
20,EDUCAÇÃO,145
44,INTERDISCIPLINAR,134
21,EDUCAÇÃO FÍSICA,109
35,ENSINO,89


## Área de Conhecimento Predominante por Campus

In [ ]:
analise_area_conhecimento_por_campus = (
    projetos_df
    .groupby(["Campus", "Área de Conhecimento"])
    .agg(total_projetos=("Título do Projeto", "count"))
    .reset_index()
    .rename(columns={"total_projetos": "Total de Projetos"})
    .sort_values(["Campus", "Total de Projetos"], ascending=[True, False])
)

analise_area_conhecimento_por_campus = (
    analise_area_conhecimento_por_campus
    .loc[analise_area_conhecimento_por_campus.groupby("Campus")["Total de Projetos"].idxmax()]
    .reset_index(drop=True)
)

display(analise_area_conhecimento_por_campus)

## Tipo de Edital por Área de Conhecimento

In [31]:
analise_edital_por_area_conhecimento = projetos_df.groupby("Área de Conhecimento").agg(
    total_projetos=("Título do Projeto", "count"),
    total_externo=("Edital", lambda x: (x == "FOMENTO EXTERNO").sum()),
    total_pibic=("Edital", lambda x: (x == "PIBIC").sum()),
    total_pibic_af=("Edital", lambda x: (x == "PIBIC AF").sum()),
    total_pibic_jr=("Edital", lambda x: (x == "PIBIC JR").sum()),
    total_pibic_jr_af=("Edital", lambda x: (x == "PIBIC JR AF").sum()),
    total_pictv=("Edital", lambda x: (x == "PICTV").sum()),
    total_pibiti=("Edital", lambda x: (x == "PIBITI").sum())
).reset_index().rename(columns={
    "total_projetos": "Total de Projetos",
    "total_externo": "Fomento Externo",
    "total_pibic": "PIBIC",
    "total_pibic_af": "PIBIC AF",
    "total_pibic_jr": "PIBIC JR",
    "total_pibic_jr_af": "PIBIC JR AF",
    "total_pictv": "PICTV",
    "total_pibiti": "PIBITI"
}).sort_values(by="Total de Projetos", ascending=False)

analise_edital_por_area_conhecimento['Fomento Externo'] = (
    analise_edital_por_area_conhecimento['Fomento Externo'].astype(str) +
    " (" + (analise_edital_por_area_conhecimento['Fomento Externo'] / analise_edital_por_area_conhecimento['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_edital_por_area_conhecimento['PIBIC'] = (
    analise_edital_por_area_conhecimento['PIBIC'].astype(str) +
      " (" + (analise_edital_por_area_conhecimento['PIBIC'] / analise_edital_por_area_conhecimento['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)  

analise_edital_por_area_conhecimento['PIBIC AF'] = (
    analise_edital_por_area_conhecimento['PIBIC AF'].astype(str) + 
    " (" + (analise_edital_por_area_conhecimento['PIBIC AF'] / analise_edital_por_area_conhecimento['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_edital_por_area_conhecimento['PIBIC JR'] = (
    analise_edital_por_area_conhecimento['PIBIC JR'].astype(str) + 
    " (" + (analise_edital_por_area_conhecimento['PIBIC JR'] / analise_edital_por_area_conhecimento['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_edital_por_area_conhecimento['PIBIC JR AF'] = (
    analise_edital_por_area_conhecimento['PIBIC JR AF'].astype(str) +
      " (" + (analise_edital_por_area_conhecimento['PIBIC JR AF'] / analise_edital_por_area_conhecimento['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_edital_por_area_conhecimento['PICTV'] = (
    analise_edital_por_area_conhecimento['PICTV'].astype(str) + " (" +
      (analise_edital_por_area_conhecimento['PICTV'] / analise_edital_por_area_conhecimento['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_edital_por_area_conhecimento['PIBITI'] = (
    analise_edital_por_area_conhecimento['PIBITI'].astype(str) + " (" +
      (analise_edital_por_area_conhecimento['PIBITI'] / analise_edital_por_area_conhecimento['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

display(analise_edital_por_area_conhecimento)

,Área de Conhecimento,Total de Projetos,Fomento Externo,PIBIC,PIBIC AF,PIBIC JR,PIBIC JR AF,PICTV,PIBITI
11,CIÊNCIA DA COMPUTAÇÃO,483,72 (14.91%),109 (22.57%),32 (6.63%),56 (11.59%),12 (2.48%),109 (22.57%),93 (19.25%)
30,ENGENHARIA ELÉTRICA,293,24 (8.19%),87 (29.69%),15 (5.12%),56 (19.11%),11 (3.75%),27 (9.22%),73 (24.91%)
1,AGRONOMIA,225,4 (1.78%),78 (34.67%),35 (15.56%),35 (15.56%),7 (3.11%),24 (10.67%),42 (18.67%)
57,QUÍMICA,192,4 (2.08%),56 (29.17%),9 (4.69%),37 (19.27%),14 (7.29%),43 (22.4%),29 (15.1%)
13,CIÊNCIA E TECNOLOGIA DE ALIMENTOS,188,0 (0.0%),64 (34.04%),19 (10.11%),21 (11.17%),9 (4.79%),17 (9.04%),58 (30.85%)
34,ENGENHARIA SANITÁRIA,185,2 (1.08%),64 (34.59%),14 (7.57%),31 (16.76%),6 (3.24%),30 (16.22%),38 (20.54%)
20,EDUCAÇÃO,181,5 (2.76%),64 (35.36%),20 (11.05%),15 (8.29%),8 (4.42%),57 (31.49%),12 (6.63%)
44,INTERDISCIPLINAR,152,5 (3.29%),47 (30.92%),24 (15.79%),23 (15.13%),8 (5.26%),13 (8.55%),32 (21.05%)
21,EDUCAÇÃO FÍSICA,125,3 (2.4%),46 (36.8%),20 (16.0%),18 (14.4%),2 (1.6%),20 (16.0%),16 (12.8%)
65,ZOOTECNIA,103,5 (4.85%),33 (32.04%),10 (9.71%),19 (18.45%),8 (7.77%),16 (15.53%),12 (11.65%)


# Pesquisadores

## Pesquisadores por Ano

In [40]:
analise_pesquisadores_ano = membros_projetos_df.groupby("Ano de Início da Execução").agg(
    total_pesquisadores=("Nome do Membro", "count"),
    bolsistas=("Vínculo com Projeto", lambda x: (x == "Bolsista").sum()),
    voluntarios=("Vínculo com Projeto", lambda x: (x == "Voluntário").sum()),
).reset_index().rename(columns={"total_pesquisadores": "Total de Pesquisadores", "bolsistas": "Bolsistas", "voluntarios": "Voluntários"})

analise_pesquisadores_ano['Bolsistas'] = (
    analise_pesquisadores_ano['Bolsistas'].astype(str) +
    ' (' + (analise_pesquisadores_ano['Bolsistas'] / analise_pesquisadores_ano['Total de Pesquisadores'] * 100).round(2).astype(str) + '%)'
)

analise_pesquisadores_ano['Voluntários'] = (
    analise_pesquisadores_ano['Voluntários'].astype(str) +
    ' (' + (analise_pesquisadores_ano['Voluntários'] / analise_pesquisadores_ano['Total de Pesquisadores'] * 100).round(2).astype(str) + '%)'
)

display(analise_pesquisadores_ano)

,Ano de Início da Execução,Total de Pesquisadores,Bolsistas,Voluntários
0,2014.0,1,0 (0.0%),1 (100.0%)
1,2022.0,20,12 (60.0%),8 (40.0%)
2,2023.0,2025,614 (30.32%),1411 (69.68%)
3,2024.0,2133,648 (30.38%),1485 (69.62%)
4,2025.0,2959,847 (28.62%),2112 (71.38%)
5,2026.0,190,17 (8.95%),173 (91.05%)


## Pesquisadores por Edital

In [42]:
analise_pesquisadores_edital = membros_projetos_df.groupby("Categoria do Membro").agg(
    total_projetos=("ID do Projeto", "count"),
    total_externo=("Edital", lambda x: (x == "FOMENTO EXTERNO").sum()),
    total_pibic=("Edital", lambda x: (x == "PIBIC").sum()),
    total_pibic_af=("Edital", lambda x: (x == "PIBIC AF").sum()),
    total_pibic_jr=("Edital", lambda x: (x == "PIBIC JR").sum()),
    total_pibic_jr_af=("Edital", lambda x: (x == "PIBIC JR AF").sum()),
    total_pictv=("Edital", lambda x: (x == "PICTV").sum()),
    total_pibiti=("Edital", lambda x: (x == "PIBITI").sum())
).reset_index().rename(columns={
    "total_projetos": "Total de Projetos",
    "total_externo": "Fomento Externo",
    "total_pibic": "PIBIC",
    "total_pibic_af": "PIBIC AF",
    "total_pibic_jr": "PIBIC JR",
    "total_pibic_jr_af": "PIBIC JR AF",
    "total_pictv": "PICTV",
    "total_pibiti": "PIBITI"
})
analise_pesquisadores_edital = analise_pesquisadores_edital.sort_values(by="Total de Projetos", ascending=False)

analise_pesquisadores_edital['Fomento Externo'] = (
    analise_pesquisadores_edital['Fomento Externo'].astype(str) +
    ' (' + (analise_pesquisadores_edital['Fomento Externo'] / analise_pesquisadores_edital['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_pesquisadores_edital['PIBIC'] = (
    analise_pesquisadores_edital['PIBIC'].astype(str) + 
    ' (' + (analise_pesquisadores_edital['PIBIC'] / analise_pesquisadores_edital['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_pesquisadores_edital['PIBIC AF'] = (
    analise_pesquisadores_edital['PIBIC AF'].astype(str) +
    ' (' + (analise_pesquisadores_edital['PIBIC AF'] / analise_pesquisadores_edital['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_pesquisadores_edital['PIBIC JR'] = (
    analise_pesquisadores_edital['PIBIC JR'].astype(str) +
    ' (' + (analise_pesquisadores_edital['PIBIC JR'] / analise_pesquisadores_edital['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_pesquisadores_edital['PIBIC JR AF'] = (
    analise_pesquisadores_edital['PIBIC JR AF'].astype(str) +
    ' (' + (analise_pesquisadores_edital['PIBIC JR AF'] / analise_pesquisadores_edital['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_pesquisadores_edital['PICTV'] = (
    analise_pesquisadores_edital['PICTV'].astype(str) +
    ' (' + (analise_pesquisadores_edital['PICTV'] / analise_pesquisadores_edital['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_pesquisadores_edital['PIBITI'] = (
    analise_pesquisadores_edital['PIBITI'].astype(str) +
    ' (' + (analise_pesquisadores_edital['PIBITI'] / analise_pesquisadores_edital['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

display(analise_pesquisadores_edital)

,Categoria do Membro,Total de Projetos,Fomento Externo,PIBIC,PIBIC AF,PIBIC JR,PIBIC JR AF,PICTV,PIBITI
1,Docente,4083,679 (16.63%),1097 (26.87%),314 (7.69%),509 (12.47%),134 (3.28%),799 (19.57%),551 (13.49%)
0,Discente,2923,276 (9.44%),590 (20.18%),95 (3.25%),322 (11.02%),90 (3.08%),1264 (43.24%),286 (9.78%)
3,TAE,234,94 (40.17%),38 (16.24%),12 (5.13%),16 (6.84%),12 (5.13%),51 (21.79%),11 (4.7%)
2,Não Informado,88,38 (43.18%),3 (3.41%),0 (0.0%),17 (19.32%),5 (5.68%),23 (26.14%),2 (2.27%)


## Pesquisadores por Área de Conhecimento

In [43]:
analise_pesquisadores_area_conhecimento = membros_projetos_df.groupby("Área de Conhecimento").agg(
    total_projetos=("ID do Projeto", "count"),
    discente=("Categoria do Membro", lambda x: (x == "Discente").sum()),
    docente=("Categoria do Membro", lambda x: (x == "Docente").sum()),
    tae=("Categoria do Membro", lambda x: (x == "TAE").sum()),
    outros=("Categoria do Membro", lambda x: (x == "Não Informado").sum())
).reset_index().rename(columns={
    "total_projetos": "Total de Projetos",
    "discente": "Discente",
    "docente": "Docente",
    "tae": "TAE",
    "outros": "Não Informado"
}).sort_values(by="Total de Projetos", ascending=False)

analise_pesquisadores_area_conhecimento['Discente'] = (
    analise_pesquisadores_area_conhecimento['Discente'].astype(str) +
    ' (' + (analise_pesquisadores_area_conhecimento['Discente'] / analise_pesquisadores_area_conhecimento['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_pesquisadores_area_conhecimento['Docente'] = (
    analise_pesquisadores_area_conhecimento['Docente'].astype(str) +
    ' (' + (analise_pesquisadores_area_conhecimento['Docente'] / analise_pesquisadores_area_conhecimento['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_pesquisadores_area_conhecimento['TAE'] = (
    analise_pesquisadores_area_conhecimento['TAE'].astype(str) +
    ' (' + (analise_pesquisadores_area_conhecimento['TAE'] / analise_pesquisadores_area_conhecimento['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_pesquisadores_area_conhecimento['Não Informado'] = (
    analise_pesquisadores_area_conhecimento['Não Informado'].astype(str) +
    ' (' + (analise_pesquisadores_area_conhecimento['Não Informado'] / analise_pesquisadores_area_conhecimento['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

display(analise_pesquisadores_area_conhecimento)

,Área de Conhecimento,Total de Projetos,Discente,Docente,TAE,Não Informado
11,CIÊNCIA DA COMPUTAÇÃO,1462,555 (37.96%),793 (54.24%),81 (5.54%),33 (2.26%)
30,ENGENHARIA ELÉTRICA,657,193 (29.38%),442 (67.28%),13 (1.98%),9 (1.37%)
1,AGRONOMIA,437,171 (39.13%),238 (54.46%),21 (4.81%),7 (1.6%)
57,QUÍMICA,387,155 (40.05%),214 (55.3%),15 (3.88%),3 (0.78%)
34,ENGENHARIA SANITÁRIA,385,168 (43.64%),194 (50.39%),17 (4.42%),6 (1.56%)
20,EDUCAÇÃO,363,160 (44.08%),198 (54.55%),5 (1.38%),0 (0.0%)
13,CIÊNCIA E TECNOLOGIA DE ALIMENTOS,351,131 (37.32%),211 (60.11%),7 (1.99%),2 (0.57%)
44,INTERDISCIPLINAR,293,121 (41.3%),170 (58.02%),2 (0.68%),0 (0.0%)
21,EDUCAÇÃO FÍSICA,230,90 (39.13%),140 (60.87%),0 (0.0%),0 (0.0%)
35,ENSINO,225,99 (44.0%),123 (54.67%),1 (0.44%),2 (0.89%)


# Grupos de Pesquisa

In [16]:
total_grupos_de_pesquisa = grupos_pesquisa_df["Nome do Grupo de Pesquisa"].nunique()
print(f"Total de Grupos de Pesquisa: {total_grupos_de_pesquisa}")

total_projetos_grupo_pesquisa = grupos_pesquisa_df["Total de Projetos Aprovados"].sum()
print(f"Total de Projetos Aprovados em Grupos de Pesquisa: {total_projetos_grupo_pesquisa}")

Total de Grupos de Pesquisa: 174
Total de Projetos Aprovados em Grupos de Pesquisa: 1667


# Produção Técnica

In [ ]:
total_producoes_tecnicas = producoes_tecnicas_df.shape[0]
print(f"Total de Produções Técnicas: {total_producoes_tecnicas}")

analise_producoes_tecnicas_por_tipo = producoes_tecnicas_df.loc[:, ["Tipo de Publicação", "ID"]]
analise_producoes_tecnicas_por_tipo = analise_producoes_tecnicas_por_tipo.groupby("Tipo de Publicação").count().reset_index()
analise_producoes_tecnicas_por_tipo = analise_producoes_tecnicas_por_tipo.rename(columns={
    "ID": "Total de Produções Técnicas"
})
analise_producoes_tecnicas_por_tipo = analise_producoes_tecnicas_por_tipo.sort_values(by="Total de Produções Técnicas", ascending=False)
analise_producoes_tecnicas_por_tipo["Total de Produções Técnicas"] = (
    analise_producoes_tecnicas_por_tipo["Total de Produções Técnicas"].astype(str)
    + " ("
    + (
        analise_producoes_tecnicas_por_tipo["Total de Produções Técnicas"].astype(int)
        / total_producoes_tecnicas * 100
    ).round(2).astype(str)
    + "%)"
)
display(analise_producoes_tecnicas_por_tipo)


Total de Produções Técnicas: 82177


,Tipo de Publicação,Total de Produções Técnicas
0,Apresentação De Trabalho,35683 (43.42%)
8,Organização De Evento,16275 (19.8%)
2,Curso De Curta Duração Ministrado,10433 (12.7%)
16,Trabalho Tecnico,8208 (9.99%)
13,Programa De Radio Ou Tv,3006 (3.66%)
9,Outra Produção Tecnica,2414 (2.94%)
3,Desenvolvimento De Material Didatico Ou Instrucional,2275 (2.77%)
15,Software,1034 (1.26%)
14,Relatorio De Pesquisa,730 (0.89%)
10,Patente,542 (0.66%)


# Orientações

# Destaques